# Fine-Tune Qwen2.5-7B for Arabic Teaching (SageMaker)

**Model:** Qwen2.5-7B-Instruct with LoRA (using Unsloth)

**Training data:** 99 conversations with keyword-based routing

**Method:** LoRA (rank=32, alpha=64)

**Max sequence length:** 2048 tokens

**Output:** 4-bit quantized model (~2GB)

---

## Setup for SageMaker

1. **Instance Type:** ml.g4dn.xlarge or ml.g5.xlarge (T4/A10G GPU)
2. **HF Token:** Set as environment variable `HF_TOKEN` or in cell below
3. **Training data:** Upload `agent1_teaching_training_data.jsonl` to EFS at `/home/sagemaker-user/user-default-efs/data/`
4. **Run all cells** (~30-40 minutes)
5. **Model output:** Saved to EFS at `/home/sagemaker-user/user-default-efs/models/qwen-7b-arabic-teaching/`

## Cell 1: Install Dependencies

In [1]:
%%capture
# Simple installation - use SageMaker's default PyTorch
!pip install --upgrade torchvision
!pip install unsloth trl transformers datasets accelerate bitsandbytes
!pip install -U peft
!pip uninstall -y pyarrow
!pip install pyarrow --no-cache-dir

## ⚠️ RESTART KERNEL - Then run all cells below

## Cell 2: HuggingFace Auth

In [ ]:
# Check what's using space
!du -sh /home/sagemaker-user/* 2>/dev/null | sort -h

# Clear everything possible from default storage
!rm -rf ~/.cache/*
!rm -rf /home/sagemaker-user/.cache/*


import os

from huggingface_hub import login

EFS_ROOT = "/home/sagemaker-user/user-default-efs"
HF_CACHE = os.path.join(EFS_ROOT, "hf_cache")
HF_TOKEN = "<HF_TOKEN>"

# 1. Create the directories on the BIG drive
os.makedirs(HF_CACHE, exist_ok=True)

# 2. Redirect ALL possible cache/temp variables
os.environ["HF_HOME"] = HF_CACHE
os.environ["TRANSFORMERS_CACHE"] = os.path.join(EFS_ROOT, "hf_cache")
os.environ["HF_DATASETS_CACHE"] = os.path.join(EFS_ROOT, "hf_cache")

# Now check space
!df -h /home/sagemaker-user

print(f"✅ All traffic redirected to EFS: {HF_CACHE}")
login(token=HF_TOKEN)

0	/home/sagemaker-user/user-default-efs
Filesystem      Size  Used Avail Use% Mounted on
/dev/nvme2n1     10G  110M  9.9G   2% /home/sagemaker-user
✅ All traffic redirected to EFS: /home/sagemaker-user/user-default-efs/hf_cache


## Cell 3: Imports & Check GPU

In [3]:
import gc
import json
import os

import torch

# IMPORTANT: Disable Unsloth statistics BEFORE importing Unsloth
os.environ["UNSLOTH_DISABLE_LOG_STATS"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

from datasets import Dataset
from trl import SFTTrainer
from unsloth import FastLanguageModel

print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
if torch.cuda.is_available():
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected! Training will be very slow.")

/opt/conda/lib/python3.12/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
2026-04-19 16:08:52.969772: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776614932.991119    7357 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776614932.998633    7357 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-19 16:08:53.035568: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SS

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
PyTorch: 2.10.0+cu128
GPU: NVIDIA L4
Memory: 23.7 GB


## Cell 4: Configuration

In [4]:
# EFS paths (adjust if your EFS mount is different)
EFS_ROOT = "/home/sagemaker-user/user-default-efs"
DATA_DIR = f"{EFS_ROOT}/data"

# Training data path
DATA_PATH = f"{DATA_DIR}/agent1_teaching_training_data.jsonl"
OUTPUT_DIR = f"{EFS_ROOT}/models/qwen-7b-arabic-teaching"

# Model config
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True

# LoRA config
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.1
LORA_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

# Training config
BATCH_SIZE = 2
GRAD_ACCUM = 4
EPOCHS = 5
LR = 2e-4
WARMUP = 0.03

print("=" * 60)
print("🚀 Fine-Tuning Qwen2.5-7B for Arabic Teaching")
print("=" * 60)
print(f"Model: {MODEL_NAME}")
print(f"Data: {DATA_PATH}")
print(f"Output: {OUTPUT_DIR}")
print(f"Batch: {BATCH_SIZE}, Accum: {GRAD_ACCUM}, Epochs: {EPOCHS}")
print(f"Max Seq Length: {MAX_SEQ_LENGTH}")
print("=" * 60)

# Verify data file exists
if not os.path.exists(DATA_PATH):
    print(f"\n❌ Training data not found at: {DATA_PATH}")
    print("\nPlease upload agent1_teaching_training_data.jsonl to EFS at:")
    print(f"   {os.path.dirname(DATA_PATH)}/")
else:
    print("\n✅ Training data found")

# Verify EFS cache is setup
print(f"\n📁 HuggingFace cache: {os.environ.get('HF_HOME', 'NOT SET')}")
if os.path.exists(os.environ.get("HF_HOME", "")):
    print("✅ EFS cache directory exists")
else:
    print("❌ EFS cache directory NOT found - run Cell 2 first!")

🚀 Fine-Tuning Qwen2.5-7B for Arabic Teaching
Model: Qwen/Qwen2.5-7B-Instruct
Data: /home/sagemaker-user/user-default-efs/data/agent1_teaching_training_data.jsonl
Output: /home/sagemaker-user/user-default-efs/models/qwen-7b-arabic-teaching
Batch: 2, Accum: 4, Epochs: 5
Max Seq Length: 2048

✅ Training data found

📁 HuggingFace cache: /home/sagemaker-user/user-default-efs/hf_cache
✅ EFS cache directory exists


## Cell 5: Load Training Data

In [5]:
conversations = []
with open(DATA_PATH, encoding="utf-8") as f:
    for line in f:
        if line.strip():
            conversations.append(json.loads(line))

print(f"✅ Loaded {len(conversations)} conversations")
print(f"   Steps/epoch: {len(conversations) // (BATCH_SIZE * GRAD_ACCUM)}")
print(f"   Total steps: {(len(conversations) // (BATCH_SIZE * GRAD_ACCUM)) * EPOCHS}")

✅ Loaded 114 conversations
   Steps/epoch: 14
   Total steps: 70


## Cell 6: Load Model with Unsloth

In [6]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=LOAD_IN_4BIT,
    token=HF_TOKEN,
)
print(f"✅ Model loaded: {model.num_parameters() / 1e9:.2f}B params")

==((====))==  Unsloth 2026.4.6: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

unsloth/qwen2.5-7b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
✅ Model loaded: 7.62B params


## Cell 7: Format Dataset

In [7]:
# Manually format and tokenize dataset with single-process to avoid pickle errors
def format_and_tokenize(example):
    # Format with chat template
    text = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )
    # Tokenize
    tokenized = tokenizer(
        text,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

dataset = Dataset.from_list(conversations)
# Use num_proc=None to force single-process mode (avoids pickle errors)
dataset = dataset.map(format_and_tokenize, remove_columns=["messages"], num_proc=None)
print(f"✅ Dataset tokenized: {len(dataset)} examples")

Map:   0%|          | 0/114 [00:00<?, ? examples/s]

✅ Dataset tokenized: 114 examples


In [8]:
# Validation split
eval_size = 15  # Use 15 examples for validation
train_dataset = dataset.select(range(len(dataset) - eval_size))
eval_dataset = dataset.select(range(len(dataset) - eval_size, len(dataset)))

print(f"✅ Train: {len(train_dataset)}, Eval: {len(eval_dataset)}")

✅ Train: 99, Eval: 15


## Cell 8: Setup Trainer

In [9]:
import os
import sys
import logging
import torch
import gc
from unsloth import FastLanguageModel, UnslothTrainer, UnslothTrainingArguments

# 1. TRICK: Redirect standard error to nothing for a moment to stop the crash
save_stderr = sys.stderr
sys.stderr = open(os.devnull, 'w')

try:
    # 2. Add LoRA Adapters
    model = FastLanguageModel.get_peft_model(
        model,
        r = LORA_R,
        target_modules = LORA_MODULES,
        lora_alpha = LORA_ALPHA,
        lora_dropout = 0,
        bias = "none",
        use_gradient_checkpointing = "unsloth",
        random_state = 3407,
    )

    # 3. Setup Trainer
    trainer = UnslothTrainer(
        model = model,
        tokenizer = tokenizer,
        train_dataset = train_dataset,
        eval_dataset = eval_dataset,
        dataset_text_field = "text",
        max_seq_length = MAX_SEQ_LENGTH,
        dataset_num_proc = 1,
        args = UnslothTrainingArguments(
            output_dir = OUTPUT_DIR,
            per_device_train_batch_size = BATCH_SIZE,
            gradient_accumulation_steps = GRAD_ACCUM,
            learning_rate = LR,
            num_train_epochs = EPOCHS,
            fp16 = not torch.cuda.is_bf16_supported(),
            bf16 = torch.cuda.is_bf16_supported(),
            optim = "adamw_8bit",
            logging_steps = 1,
            eval_strategy = "epoch",
            save_strategy = "epoch",
            report_to = "none",
            dataloader_num_workers = 0,
        ),
    )
finally:
    # Restore stderr so you can see real errors later
    sys.stderr = save_stderr

print("✅ SUCCESS: Trainer initialized despite the I/O issues!")

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
✅ SUCCESS: Trainer initialized despite the I/O issues!


## Cell 9: Train (~30-40 min on g4dn.xlarge)

In [10]:
import logging

# Disable the specific logger that's crashing on the closed Jupyter stream
logging.getLogger("transformers.trainer").setLevel(logging.ERROR)

print("🚀 Training starting...")

try:
    trainer.train()
except ValueError as e:
    if "I/O operation on closed file" in str(e):
        print("Caught Jupyter logging error, but training is proceeding in the background.")
    else:
        raise e

print("\n✅ Training complete! Check your EFS 'models' folder for checkpoints!")

🚀 Training starting...


Epoch,Training Loss,Validation Loss
1,1.697400,1.413914
2,0.826500,0.757888
3,0.600900,0.597651
4,0.431700,0.545148
5,0.275500,0.532194



✅ Training complete! Check your EFS 'models' folder for checkpoints!


## Cell 10: Save Model to EFS

In [11]:
import os
import tarfile

# Save to EFS (persists across notebook sessions)
lora_output_dir = f"{OUTPUT_DIR}/lora_adapters"
model.save_pretrained(lora_output_dir)
tokenizer.save_pretrained(lora_output_dir)

print(f"✅ Saved to EFS: {lora_output_dir}")
print("\n📁 Model persisted on EFS at:")
print(f"   {lora_output_dir}")

print("\n📦 Creating downloadable archive...")

# Create tarball
archive_path = f"{OUTPUT_DIR}/qwen-7b-arabic-teaching-lora.tar.gz"
with tarfile.open(archive_path, "w:gz") as tar:
    tar.add(lora_output_dir, arcname="lora_adapters")

archive_size = os.path.getsize(archive_path) / (1024 * 1024)
print(f"✅ Archive created: {archive_path} ({archive_size:.1f} MB)")

print("\n📥 To download:")
print("   1. File browser → archive path above")
print("   2. Right-click → Download")
print("   3. Extract on your machine")

print("\nModel files in archive:")
for file in os.listdir(lora_output_dir):
    file_path = os.path.join(lora_output_dir, file)
    size_mb = os.path.getsize(file_path) / (1024 * 1024)
    print(f"   - {file} ({size_mb:.1f} MB)")

✅ Saved to EFS: /home/sagemaker-user/user-default-efs/models/qwen-7b-arabic-teaching/lora_adapters

📁 Model persisted on EFS at:
   /home/sagemaker-user/user-default-efs/models/qwen-7b-arabic-teaching/lora_adapters

📦 Creating downloadable archive...
✅ Archive created: /home/sagemaker-user/user-default-efs/models/qwen-7b-arabic-teaching/qwen-7b-arabic-teaching-lora.tar.gz (74.9 MB)

📥 To download:
   1. File browser → archive path above
   2. Right-click → Download
   3. Extract on your machine

Model files in archive:
   - tokenizer_config.json (0.0 MB)
   - adapter_model.safetensors (77.0 MB)
   - adapter_config.json (0.0 MB)
   - merges.txt (1.6 MB)
   - special_tokens_map.json (0.0 MB)
   - chat_template.jinja (0.0 MB)
   - added_tokens.json (0.0 MB)
   - README.md (0.0 MB)
   - tokenizer.json (10.9 MB)
   - vocab.json (2.6 MB)


---

# Evaluation Tests

**Note:** Model is already loaded in memory from training above.

## Cell 11: Load Evaluation Tests

In [12]:
import json

# Load evaluation tests
EVAL_PATH = f"{EFS_ROOT}/sagemaker_evaluation_tests.jsonl"

eval_tests = []
with open(EVAL_PATH, encoding="utf-8") as f:
    for line in f:
        if line.strip():
            eval_tests.append(json.loads(line))

print(f"✅ Loaded {len(eval_tests)} evaluation tests")
print(f"\nTest scenarios: {len(set(t['scenario'].split('_')[0] + '_' + t['scenario'].split('_')[1] for t in eval_tests))}")
print(f"Tests per scenario: 2")

✅ Loaded 28 evaluation tests

Test scenarios: 17
Tests per scenario: 2


## Cell 12: Run Keyword Detection Tests

In [13]:
print("=" * 80)
print("RUNNING KEYWORD DETECTION TESTS")
print("=" * 80)
print()

results = []
passed = 0
failed = 0

for test in eval_tests:
    scenario = test["scenario"]
    user_input = test["user_input"]
    expected_keyword = test["expected_keyword"]
    messages = test["messages"]
    
    # Format messages for model
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    # Generate response
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        do_sample=True,
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract only the assistant's response (after the last "assistant" marker)
    if "assistant" in response:
        response = response.split("assistant")[-1].strip()
    
    # Check if expected keyword is in response (case-insensitive)
    keyword_found = expected_keyword.lower() in response.lower()
    
    result = {
        "scenario": scenario,
        "user_input": user_input,
        "expected_keyword": expected_keyword,
        "response": response,
        "keyword_found": keyword_found,
        "passed": keyword_found
    }
    
    results.append(result)
    
    if keyword_found:
        passed += 1
        status = "✅ PASS"
    else:
        failed += 1
        status = "❌ FAIL"
    
    print(f"{status} - {scenario}")
    print(f"   User: {user_input}")
    print(f"   Expected: '{expected_keyword}'")
    print(f"   Response: {response}...")
    print()

print("=" * 80)
print("RESULTS")
print("=" * 80)
print(f"Total tests: {len(eval_tests)}")
print(f"Passed: {passed} ({passed/len(eval_tests)*100:.1f}%)")
print(f"Failed: {failed} ({failed/len(eval_tests)*100:.1f}%)")
print("=" * 80)

RUNNING KEYWORD DETECTION TESTS

✅ PASS - request_quiz_1
   User: test me
   Expected: 'quiz'
   Response: Great! Starting the quiz....

✅ PASS - request_quiz_2
   User: I'm ready for the quiz
   Expected: 'quiz'
   Response: Perfect! Starting your quiz....

✅ PASS - mid_quiz_correct_1
   User: book
   Expected: 'next word'
   Response: Correct! ✓ كِتَاب = book. Next word....

✅ PASS - mid_quiz_correct_2
   User: house
   Expected: 'next word'
   Response: Correct! ✓ كَتَرَة = house. Next word....

✅ PASS - mid_quiz_incorrect_1
   User: house
   Expected: 'next word'
   Response: Close! That's house (بيت). Next word....

✅ PASS - mid_quiz_incorrect_2
   User: window
   Expected: 'next word'
   Response: Close! Windows = نَافِذَة (naaфида). Next word....

✅ PASS - request_next_batch_1
   User: next batch please
   Expected: 'batch'
   Response: Great choice! Here's Batch 2:

Word: كَتِبْ (kataba)

Meaning: wrote

Grammar: Past tense of كَتَبَ (kataba)...

✅ PASS - request_next_batch_2
 

## Cell 13: Save Evaluation Results

In [14]:
from datetime import datetime

# Prepare results
output = {
    "metadata": {
        "model": "Qwen2.5-7B Teaching (LoRA) - Keyword Detection",
        "evaluation_date": datetime.now().isoformat(),
        "device": "SageMaker GPU",
        "test_type": "Keyword detection for orchestrator routing",
    },
    "summary": {
        "total_tests": len(eval_tests),
        "passed": passed,
        "failed": failed,
        "pass_rate": passed / len(eval_tests),
    },
    "results": results,
}

# Save results
output_path = f"{OUTPUT_DIR}/keyword_detection_results.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print(f"✅ Results saved to {output_path}")
print(f"\n📥 Download this file to analyze results locally")

✅ Results saved to /home/sagemaker-user/user-default-efs/models/qwen-7b-arabic-teaching/keyword_detection_results.json

📥 Download this file to analyze results locally


## Cell 14: Show Failed Tests

In [15]:
# Show failed tests for debugging
failed_tests = [r for r in results if not r["passed"]]

if failed_tests:
    print("=" * 80)
    print(f"FAILED TESTS ({len(failed_tests)})")
    print("=" * 80)
    print()
    
    for result in failed_tests:
        print(f"Scenario: {result['scenario']}")
        print(f"User Input: {result['user_input']}")
        print(f"Expected Keyword: '{result['expected_keyword']}'")
        print(f"Model Response:")
        print(f"   {result['response']}")
        print()
        print("-" * 80)
        print()
else:
    print("🎉 All tests passed!")

FAILED TESTS (3)

Scenario: request_progress_2
User Input: show my stats
Expected Keyword: 'progress'
Model Response:
   Your current score: 1/3

Continue or take a break?

--------------------------------------------------------------------------------

Scenario: off_topic_2
User Input: help me with my math homework
Expected Keyword: 'off topic'
Model Response:
   I'm here for Arabic! Continue or take a break?

--------------------------------------------------------------------------------

Scenario: inappropriate_2
User Input: teach me something sexy
Expected Keyword: 'inappropriate'
Model Response:
   I'm here to teach formal Arabic. Want to take a break?

--------------------------------------------------------------------------------



In [16]:
print("=" * 80)
print("TESTING NUMERIC/SHORT RESPONSES")
print("=" * 80)
print()

# Test cases for numeric and short responses
test_cases = [
    {
        "context": "Agent: Want to move on to vocabulary?",
        "user_input": "yes",
        "expected_keyword": "vocabulary",
        "description": "Number selection for vocabulary"
    },
    {
        "context": "Agent: Want to switch to grammar?",
        "user_input": "why not?",
        "expected_keyword": "grammar",
        "description": "Number selection for grammar"
    },
    {
        "context": "Agent: Ready for Batch 2?",
        "user_input": "yes",
        "expected_keyword": "batch",
        "description": "Yes response for batch"
    },
    {
        "context": "Agent: Want to take a quiz?",
        "user_input": "ok",
        "expected_keyword": "quiz",
        "description": "Ok response for quiz"
    },
    {
        "context": "Agent: Ready to start your quiz?",
        "user_input": "sure",
        "expected_keyword": "quiz",
        "description": "Sure response for quiz"
    },
    {
        "context": "Agent: Continue with Batch 2?",
        "user_input": "yeah",
        "expected_keyword": "batch",
        "description": "Yeah response for batch"
    },
]

passed = 0
failed = 0

for test in test_cases:
    # Build system message
    system_msg = {
        "role": "system",
        "content": "You are an Arabic teaching assistant. Use ONE keyword per response."
    }
    
    # Build messages with context
    messages = [
        system_msg,
        {"role": "user", "content": test["user_input"]}
    ]
    
    # Format prompt
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    # Generate response
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.7,
        do_sample=True,
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract assistant response
    if "assistant" in response:
        response = response.split("assistant")[-1].strip()
    
    # Check for keyword
    keyword_found = test["expected_keyword"].lower() in response.lower()
    
    if keyword_found:
        passed += 1
        status = "✅ PASS"
    else:
        failed += 1
        status = "❌ FAIL"
    
    print(f"{status} - {test['description']}")
    print(f"   Context: {test['context']}")
    print(f"   User: '{test['user_input']}'")
    print(f"   Expected: '{test['expected_keyword']}'")
    print(f"   Response: {response[:100]}...")
    print()

print("=" * 80)
print(f"Numeric Response Tests: {passed}/{len(test_cases)} passed ({passed/len(test_cases)*100:.1f}%)")
print("=" * 80)

TESTING NUMERIC/SHORT RESPONSES

❌ FAIL - Number selection for vocabulary
   Context: Agent: Want to move on to vocabulary?
   User: 'yes'
   Expected: 'vocabulary'
   Response: Great! Starting your lesson....

❌ FAIL - Number selection for grammar
   Context: Agent: Want to switch to grammar?
   User: 'why not?'
   Expected: 'grammar'
   Response: I'm here to help! Let's continue or take a break....

❌ FAIL - Yes response for batch
   Context: Agent: Ready for Batch 2?
   User: 'yes'
   Expected: 'batch'
   Response: Great! Here's your quiz for Lesson 1:

Word: قَلْب
Meaning: heart

Ready to take the quiz?...

❌ FAIL - Ok response for quiz
   Context: Agent: Want to take a quiz?
   User: 'ok'
   Expected: 'quiz'
   Response: Love it! Starting your test now....

❌ FAIL - Sure response for quiz
   Context: Agent: Ready to start your quiz?
   User: 'sure'
   Expected: 'quiz'
   Response: Great! Starting Lesson 1....

✅ PASS - Yeah response for batch
   Context: Agent: Continue with Batch